# Importing libraries

In [ ]:
import foldersfiles as ff
import functions as fn

import numpy as np
import pandas as pd
import math

import json

from pathlib import Path

import pickle

# Importing data

In [ ]:
df = fn.import_func(
    ff.CLEANED_ORIGINALS_DATA_DIR / ff.PARQUETS / f'{ff.EUMACO}-all.parquet'
)

Drop `json_id` and `transm_id` columns

In [ ]:
# df = df[[
#     'pseudopaz' ,
#     # 'json_id' ,
#     # 'transm_id' ,
#     'datetime' ,
#     'systbp[mmHg]' ,
#     'diastbp[mmHg]' ,
#     'map[mmHg]' ,
#     'bpthr[bpm]' ,
#     'spo2[%]' ,
#     'spo2hr[bpm]' ,
#     'weight[kg]' ,
# ]]

# Analysis

In [ ]:
df.info()

## Conflicts and merge candidates

In [ ]:
def check_measurements(df, thresholds):
    """
    thresholds:
        {measurement_column: (maximum_time_difference,
                              maximum_acceptable_value_difference)}

    Returns all pairs within the time threshold, classified by value difference.
    Missing measurements are ignored. The input dataframe is not modified.

    ag8l
    checks if two values stays within the same time threshold.
    if yes, checks the value delta:
        - if value delta is greater than the value threshold, marks as a conflict
        - if value delta is lesser than the value threshold, marks as a merge candidate

    in case of more than 3 valid rows, it shows an entry for all of the couples, so like triangular numbers.
    If n=number of duplicated rows, it shows ((n*(n-1))/2)
    """
    columns = [
        "pseudopaz", "json_id", "transm_id", "measurement", "values",
        "status", "datetimes", "time_delta", "value_delta", "row_indices",
    ]
    results = []

    def unique_ids(a, b):
        return (a,) if a == b else (a, b)

    for measurement, (time_limit, value_limit) in thresholds.items():
        time_limit = pd.Timedelta(seconds=time_limit)

        if pd.isna(time_limit) or time_limit < pd.Timedelta(0):
            raise ValueError(f"Invalid time threshold for {measurement}")
        if pd.isna(value_limit) or value_limit < 0:
            raise ValueError(f"Invalid value threshold for {measurement}")

        data = df[
            ["pseudopaz", "json_id", "transm_id", "datetime", measurement]
        ].dropna(subset=[measurement]).copy()

        data["_row_index"] = data.index
        data = data.sort_values("datetime", kind="stable")

        for patient, group in data.groupby("pseudopaz", sort=False):
            rows = list(
                group[
                    ["datetime", measurement, "json_id",
                     "transm_id", "_row_index"]
                ].itertuples(index=False, name=None)
            )

            for i, (t1, v1, json1, transm1, index1) in enumerate(rows):
                for j in range(i + 1, len(rows)):
                    t2, v2, json2, transm2, index2 = rows[j]
                    time_delta = t2 - t1

                    # Later rows are even farther away in time.
                    if time_delta > time_limit:
                        break

                    value_delta = abs(v2 - v1)

                    if value_delta == 0:
                        status = "duplicate"
                    elif value_delta <= value_limit or math.isclose(
                        value_delta, value_limit, rel_tol=0, abs_tol=1e-12
                    ):
                        status = "within_tolerance"
                    else:
                        status = "conflict"

                    results.append({
                        "pseudopaz": patient,
                        "json_id": unique_ids(json1, json2),
                        "transm_id": unique_ids(transm1, transm2),
                        "measurement": measurement,
                        "values": (v1, v2),
                        "status": status,
                        "datetimes": (t1, t2),
                        "time_delta": time_delta,
                        "value_delta": value_delta,
                        "row_indices": (index1, index2),
                    })

    return pd.DataFrame(results, columns=columns)

In [ ]:
thresholds = {
    # Measurement:       (seconds, value threshold)
    "weight [kg]":       (0, 0),
    "spo2 [%]":          (0, 0),
    "spo2 hr [bpm]":     (0, 0),
    "syst bp [mmHg]":    (0, 0),
    "diast bp [mmHg]":   (0, 0),
    "map [mmHg]":        (0, 0),
    "bpt hr [bpm]":      (0, 0),

    # "weight [kg]":       (0, 6),
    # "spo2 [%]":          (0, 1.0),
    # "spo2 hr [bpm]":     (0, 3.0),
    # "syst bp [mmHg]":    (0, 5.0),
    # "diast bp [mmHg]":   (0, 5.0),
    # "map [mmHg]":        (0, 5.0),
    # "bpt hr [bpm]":      (0, 3.0),

    # "weight [kg]":       (300, 0.2),
    # "spo2 [%]":          (60, 1.0),
    # "spo2 hr [bpm]":     (60, 3.0),
    # "syst bp [mmHg]":    (120, 5.0),
    # "diast bp [mmHg]":   (120, 5.0),
    # "map [mmHg]":        (120, 5.0),
    # "bpt hr [bpm]":      (120, 3.0),
}

if True:
    pairs = check_measurements(df, thresholds)

    # Differences exceeding the allowed value threshold:
    conflicts = pairs.loc[pairs["status"].eq("conflict")].reset_index(drop=True)

    # Duplicates and nearby values within tolerance:
    merge_candidates = pairs.loc[
        pairs["status"].isin(["duplicate", "within_tolerance"])
    ].reset_index(drop=True)

In [ ]:
conflicts
# merge_candidates

## Check of the two heartrate fields

Matches each spo2-hr reading to the nearest bpt-hr reading for the same patient within x minutes, allowing a bpt reading to be reused. It flags matched pairs that differ by more than y bpm.

In [ ]:
MAX_TIME_DIFF = pd.Timedelta(minutes=20)
MAX_BPM_DIFF = 10

spo2 = df.loc[df['spo2 hr [bpm]'].notna(), ['pseudopaz', 'datetime', 'spo2 hr [bpm]']]
bpt = df.loc[df['bpt hr [bpm]'].notna(), ['pseudopaz', 'datetime', 'bpt hr [bpm]']]

spo2 = spo2.sort_values('datetime')
bpt = bpt.sort_values('datetime')

pairs = pd.merge_asof(
    spo2,
    bpt.rename(columns={'datetime': 'datetime_bpt'}).assign(datetime=bpt['datetime']),
    on='datetime',
    by='pseudopaz',
    direction='nearest',
    tolerance=MAX_TIME_DIFF,
)

pairs = pairs.dropna(subset=['bpt hr [bpm]'])
pairs['time_diff'] = (pairs['datetime'] - pairs['datetime_bpt']).abs()
pairs['bpm_diff'] = (pairs['spo2 hr [bpm]'] - pairs['bpt hr [bpm]']).abs()

conflicts = pairs[pairs['bpm_diff'] > MAX_BPM_DIFF]

print(f'Pairs compared: {len(pairs)}')
print(f'Conflicts: {len(conflicts)} ({len(conflicts) / len(pairs):.1%})')
print(pairs['bpm_diff'].describe())
print(conflicts.sort_values('bpm_diff', ascending=False).head(20))

## Simplify the dataframe

See docstring. For example:

**`original_df`**

| pseudopaz | datetime | weight [kg] | spo2 [%] |
|---|---|---:|---:|
| A | 2026-01-01 09:00 | 70 | — |
| A | 2026-01-01 09:00 | — | 98 |
| B | 2026-01-01 10:00 | 80 | 96 |
| B | 2026-01-01 10:00 | 82 | 96 |

**`simplified_df`**

| pseudopaz | datetime | weight [kg] | spo2 [%] |
|---|---|---:|---:|
| A | 2026-01-01 09:00 | 70 | 98 |
| B | 2026-01-01 10:00 | 80 | 96 |
| B | 2026-01-01 10:00 | 82 | 96 |

In [ ]:
def simplify_measurements(df):
    '''Groups rows by patient (`pseudopaz`) and timestamp (`datetime`),
    merging each group into one row when every measurement column
    contains at most one distinct nonmissing value;
    it fills measurements from the available values
    and retains other fields from the first row.
    If any measurement has conflicting values,
    it preserves all rows in that group,
    then sorts the result by patient and timestamp and resets the index.
    '''
    measurement_cols = [
        "weight [kg]",
        "spo2 [%]",
        "spo2 hr [bpm]",
        "syst bp [mmHg]",
        "diast bp [mmHg]",
        "map [mmHg]",
        "bpt hr [bpm]",
    ]

    data = df.copy()
    rows = []

    for _, group in data.groupby(
        ["pseudopaz", "datetime"],
        sort=False,
        dropna=False,
    ):
        has_conflict = (
            group[measurement_cols]
            .nunique(dropna=True)
            .gt(1)
            .any()
        )

        if has_conflict:
            rows.extend(group.to_dict("records"))
            continue

        merged = group.iloc[0].to_dict()

        for col in measurement_cols:
            values = group[col].dropna()
            merged[col] = (
                values.iloc[0] if not values.empty else float("nan")
            )

        rows.append(merged)

    return (
        pd.DataFrame(rows, columns=data.columns)
        .sort_values(["pseudopaz", "datetime"], kind="stable")
        .reset_index(drop=True)
    )

In [ ]:
simplified_df = simplify_measurements(df)

In [ ]:
df.head()

In [ ]:
simplified_df.head()

In [ ]:
# fn.export_func(df_, ff.CLEANED_PROCESSED_DATA_DIR, '', filetype=[
#     # 'parquet' ,
#     # 'pickle' ,
#     'excel'
# ])

## Old - unknown

In [ ]:
# Count distinct non-null values within each patient/timestamp group.
distinct_counts = grouped.nunique(dropna=True)

# Identify the specific measurement columns containing conflicts.
conflict_keys = (
    distinct_counts
    .rename_axis(columns="measurement")
    .stack()
    .loc[lambda counts: counts > 1]
    .rename("distinct_values")
    .reset_index()
)

if not conflict_keys.empty:
    # Convert the original measurements to long format for inspection.
    long_data = (
        data.rename_axis("original_index")
        .reset_index()
        .melt(
            id_vars=["original_index", "pseudopaz", "datetime"],
            value_vars=measurement_cols,
            var_name="measurement",
            value_name="value",
        )
        .dropna(subset=["value"])
    )

    # Retain only values belonging to conflicting groups/columns.
    occurrences = long_data.merge(
        conflict_keys[["pseudopaz", "datetime", "measurement"]],
        on=["pseudopaz", "datetime", "measurement"],
        how="inner",
        validate="many_to_one",
    )

    conflict_details = (
        occurrences
        .groupby(["pseudopaz", "datetime", "measurement"], sort=True)
        .agg(
            values=("value", list),
            original_indices=("original_index", list),
        )
        .reset_index()
    )

    print("\nConflicting measurements:")
    print(conflict_details.to_string(index=False))

    raise ValueError(
        f"Found {len(conflict_details)} conflicting "
        "patient/timestamp/measurement combinations. "
        "Inspect conflict_details before continuing."
    )

In [ ]:
conflict_details